# Instruction-Tuning with Large Language Models (LLMs)

Instruction-based fine-tuning, commonly known as *Instruction GPT*, is a training approach where language models are optimized to understand and follow human instructions accurately. The objective is to help the model generate meaningful and task-specific responses based on user prompts.

In instruction-tuning, the dataset is highly important because it provides structured examples containing instructions, optional context, and expected outputs. By learning from these examples, the model becomes capable of handling a wide range of real-world tasks more effectively.

Many Instruction GPT systems are further improved using human feedback to refine response quality and alignment. However, this project focuses only on the instruction fine-tuning process and does not include reinforcement learning or human feedback optimization.

The instruction and context are typically merged into a single input sequence so the model can better understand the task and generate an appropriate response.

## Key Components

- **Instruction**  
  A task description or command that tells the model what action to perform.

- **Context**  
  Additional background information or supporting details required to complete the task correctly.

- **Combined Input**  
  The instruction and context are combined into one sequence before being passed to the model during training or inference.


### **Import Libraries**

In [46]:
import torch
import pickle
import io
import evaluate
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset 
from torch.utils.data.dataset import Dataset
from urllib.request import urlopen
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import LoraConfig, TaskType, get_peft_model


### **Load Dataset**

In [2]:
SEED = 42

In [3]:
dataset = load_dataset("sahil2801/CodeAlpaca-20k")

In [4]:
dataset.column_names

{'train': ['output', 'instruction', 'input']}

In [5]:
dataset['train'][1000]

{'output': 's = "Hello world" \ns = s[::-1] \nprint(s)',
 'instruction': 'Reverse the string given in the input',
 'input': 'Hello world'}

In [6]:
dataset = dataset['train']
dataset[1000]

{'output': 's = "Hello world" \ns = s[::-1] \nprint(s)',
 'instruction': 'Reverse the string given in the input',
 'input': 'Hello world'}

**Filter the Dataset Which Don't Have Any Input**

In [7]:
dataset = dataset.filter(lambda example: example['input'] == '')
dataset[0:2]

{'output': ['arr = [2, 4, 6, 8, 10]',
  'Height of triangle = opposite side length * sin (angle) / side length'],
 'instruction': ['Create an array of length 5 which contains all even numbers between 1 and 10.',
  'Formulate an equation to calculate the height of a triangle given the angle, side lengths and opposite side length.'],
 'input': ['', '']}

In [8]:
### shuffle the dataset
dataset = dataset.shuffle(seed=SEED)

In [9]:
dataset

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 9764
})

In [10]:
dataset['instruction'][0]

'Create a function that produces input strings for a calculator.'

**Split Dataset**

In [11]:
train_test_split = dataset.train_test_split(test_size=0.2,seed=SEED)
dataset_train = train_test_split['train']
dataset_test = train_test_split['test']
train_test_split

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 7811
    })
    test: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 1953
    })
})

### **Model & Tokenizer Implementation**

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m", padding_side = 'left')

In [14]:
tokenizer

GPT2Tokenizer(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [15]:
tokenizer.eos_token

'</s>'

In [16]:
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
model.to(device)

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 14924.57it/s]


OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

### Data Pre-Processing 

##### Formatting the prompt

In [17]:
def fromatting_prompt_with_response (mydataset):
    response = []
    
    for i in range(len(mydataset)):
        text = (
            f"### Instruction: \n{mydataset['instruction'][i]}"
            f"\n\n ### Response: \n{mydataset['output'][i]}</s>"
        )
        
        response.append(text)
    return response

In [18]:
def formatting_prompt_without_response(mydataset):
    
    response = []
    
    for i in range(len(mydataset)):
        text = (
            f"### Instruction: \n{mydataset['instruction'][i]}"
            f"\n\n### Response:\n"
        )
        
        response.append(text)
    return response

In [19]:
tokenized_text = tokenizer("Hi I am vishwa", return_tensors="pt")
tokenized_text

{'input_ids': tensor([[    2, 30086,    38,   524,   748,  1173,  2739]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

In [20]:
tokenized_text['input_ids'][0]

tensor([    2, 30086,    38,   524,   748,  1173,  2739])

In [21]:
expected_outputs = []

instruction_with_response = fromatting_prompt_with_response(dataset_test)
instruction_without_response = formatting_prompt_without_response(dataset_test)

for i in tqdm(range(len(instruction_with_response))):
    tokenized_instruction_with_response = tokenizer(instruction_with_response[i], return_tensors="pt")
    tokenized_instruction_without_response = tokenizer(instruction_without_response[i], return_tensors='pt')
    
    expected_output = tokenizer.decode(tokenized_instruction_with_response['input_ids'][0][len(tokenized_instruction_without_response['input_ids'][0])-1:],
                                       special_tokens = True)    
    expected_outputs.append(expected_output)
    

100%|██████████| 1953/1953 [00:01<00:00, 1131.00it/s]


In [22]:
instruction_without_response[0]

'### Instruction: \nName the most important benefit of using a database system.\n\n### Response:\n'

In [23]:
instruction_with_response[0]

'### Instruction: \nName the most important benefit of using a database system.\n\n ### Response: \nThe most important benefit of using a database system is the ability to store and retrieve data quickly and easily. Database systems also provide support for data security, data integrity, and concurrently accessing and modifying data from multiple systems.</s>'

In [24]:
expected_outputs[0]

'\nThe most important benefit of using a database system is the ability to store and retrieve data quickly and easily. Database systems also provide support for data security, data integrity, and concurrently accessing and modifying data from multiple systems.</s>'

# Converting Instructions into a PyTorch Dataset

Instead of keeping the instructions as a regular Python list, it is beneficial to convert them into a PyTorch `Dataset`. PyTorch models and training pipelines are designed to work efficiently with datasets that follow the `torch.utils.data.Dataset` structure.

To achieve this, a custom class called `ListDataset` is created by inheriting from `Dataset`. This class converts the `instructions` list into a dataset object that can be used directly in PyTorch workflows.

The `ListDataset` class typically implements:

- `__len__()` – Returns the total number of samples in the dataset.
- `__getitem__()` – Retrieves a specific sample using its index.

After converting the list into a `Dataset`, the data can be easily:

- Batched
- Shuffled
- Loaded efficiently with `DataLoader`
- Integrated into training and inference pipelines

This approach improves scalability, organization, and compatibility with industry-standard deep learning workflows in PyTorch.

In [25]:
class TemplateDataset(Dataset):
    
    def __init__(self, original_list):
        super().__init__()
        self.list = original_list
        
    def __len__(self):
        return len(self.list)

    def __getitem__(self, index):
        return self.list[index]

In [26]:
instructions = TemplateDataset(instruction_without_response)
instructions[0]

'### Instruction: \nName the most important benefit of using a database system.\n\n### Response:\n'

#### **Test Base Model Using Pipeline**

In [27]:
### define pipeline
pipeline_base_model = pipeline(task="text-generation",
                               model=model,
                               tokenizer=tokenizer,
                               batch_size = 2,
                               return_full_text = False)

In [28]:
with torch.no_grad():
    pipeline_iter = pipeline_base_model(instructions[:10], do_sample = True, temperature = 0.7, top_p =0.9,
                                        max_new_tokens = 100)

pipeline_iter

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both 

[[{'generated_text': '\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the'}],
 [{'generated_text': "You should just be able to type '#'\n\n### Instruction:\nYou should just be able to type '#'\n\n### Response:\nYou should just be able to type '#'\n\n### Instruction:\nYou should just be able to type '#'\n\n### Response:\nYou should just be able to type '#'\n\n### Instruction:\nYou should just be able to type '#'\n\n### Response:\nYou should just be"}],
 [{'generated_text': 'The first line of the text file is an empty string with the following properties:\n1. The last item is in the list.\n2. The list is empty.\n3. 

In [29]:
generate_text = []

for text in pipeline_iter:
    generate_text.append(text[0]['generated_text'])

In [30]:
generate_text[0]

'\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the most important benefit of using a database system.\n\n### Response:\n\n### Instruction:\n\nName the'

In [31]:
url = urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VvQRrSqS1P0_GobqtL-SKA/instruction-tuning-generated-outputs-base.pkl')

In [32]:
generated_texts = pickle.load(io.BytesIO(url.read()))

In [33]:
generated_texts[1]

'\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to solve an equation of the form ax + b = 0. Write corresponding code in Python.\n\n### Response:\n\n### Instruction:\nDescribe a method to

### **Model Evaluation (Pre-Taining Phase)**

# BLEU Score Evaluation

To evaluate the quality of generated responses against the expected (ground-truth) responses in the test environment, we use an automatic evaluation metric called the **BLEU score**.

BLEU (Bilingual Evaluation Understudy) is a metric originally designed to evaluate machine translation systems by measuring how closely a generated text matches one or more reference texts.

---

## How BLEU Score Works

The BLEU score compares:
- **Generated output (model prediction)**
- **Expected output (reference answer)**

It evaluates similarity based on overlapping n-grams (word sequences) between the two texts.

For each sample, a BLEU score is computed individually, and the final score is obtained by averaging across all test samples.

---

## Score Range

Depending on implementation, BLEU scores are expressed in:

- **0 to 1** (standard form), or  
- **0 to 100** (scaled version used in many implementations)

A **higher BLEU score indicates better alignment** between the generated output and the expected response.

---

## Important Notes

1. **Original Purpose**
   
   BLEU was originally designed for machine translation tasks. While it is commonly used in NLP evaluation, it may not always fully capture the quality of instruction-tuned language models. However, it still provides a useful signal for comparing generated outputs with reference answers.

---

2. **Metric Variability**

   BLEU is a **parametrized metric**, meaning its results can vary depending on:
   - n-gram size
   - smoothing techniques
   - tokenization rules

   Because of this, BLEU scores may not be directly comparable across different studies or implementations.

   To address this issue, a standardized version called **SacreBLEU** is often used. It ensures consistent evaluation by fixing the metric configuration.

   Learn more: https://aclanthology.org/W18-6319/

---

3. **Experimental Variation**

   Slight differences in BLEU scores may occur due to:
   - model randomness
   - tokenization differences
   - evaluation setup variations

   This is normal and expected when evaluating generative models.

---

## Summary

BLEU provides a fast and simple way to measure overlap between generated and expected text, making it useful for:
- baseline evaluation
- model comparison
- tracking training improvements

However, it should ideally be used alongside other evaluation methods for a more complete assessment of model quality.

In [35]:
sacrebleu_matrix = evaluate.load("sacrebleu")

results_matrix = sacrebleu_matrix.compute(predictions=generated_texts, references=expected_outputs)


In [36]:
results_matrix

{'score': 0.009296734954133449,
 'counts': [6353, 123, 5, 1],
 'totals': [481137, 479199, 477262, 475325],
 'precisions': [1.3204139361553986,
  0.025667833196646905,
  0.0010476425946335556,
  0.00021038236995739757],
 'bp': 1.0,
 'sys_len': 481137,
 'ref_len': 118330}

In [37]:
print(list(results_matrix.keys()))
print(round(results_matrix["score"], 1))

['score', 'counts', 'totals', 'precisions', 'bp', 'sys_len', 'ref_len']
0.0


In [41]:
print(f"number of model parameters: { sum(param.numel() for param in model.parameters())}")

number of model parameters: 331196416


### **Model Training**

To save time, we can perform instruction fine-tuning using a parameter-efficient fine-tuning (PEFT) technique known as Low-Rank Adaptation (LoRA).

First, we transform the base model into a PEFT-compatible model by creating a LoraConfig object from the peft library, where we specify key LoRA settings such as the rank and the target layers to be adapted. After defining this configuration, we apply it to the model using get_peft_model(), which converts the original model into a LoRA-enhanced version ready for efficient fine-tuning.

In [45]:
lora_config = LoraConfig(
    task_type= TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.1
)

In [47]:
## define peft model
peft_model = get_peft_model(model, lora_config)

W0516 12:50:46.118000 11932 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
